# Stage 4 — Four CPU CatBoost variants and full-data refits

**Question:** can stronger tree models (not new blending tricks) move the score?

- Four ordered-boosting CatBoost variants (base, extra features, slow learning rate, deeper with 3-way category combinations).
- Test predictions mix fold models (80% of data each) with models refit on 100% of the training rows.
- Blend weights fitted with constrained SLSQP on the simplex.

| result | CV logloss | public LB | wrong rows |
|---|---|---|---|
| prob_blend (fold + full) | 0.27729 | 0.109062 | 1100 |

**Finding:** best CV so far, but within leaderboard noise of stage 2. The deep CatBoost took 47% of the blend; tree models were saturating.

**Runs on Kaggle** (GPU T4 x2, Internet on for `pip install`). Inputs and the saved-prediction chain are described in the [README](../README.md#reproducing).

In [1]:
import os, glob, time, subprocess, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

def find_file(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(name)
    return os.path.abspath(hits[0]) if hits else None

def gpu_available():
    try:
        return subprocess.run(['nvidia-smi'], capture_output=True).returncode == 0
    except FileNotFoundError:
        return False

TRAIN_PATH, TEST_PATH = find_file('Train.csv'), find_file('Test.csv')
if not TRAIN_PATH:
    raise FileNotFoundError('Train.csv not found - attach the dataset via "Add Input".')
OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
USE_GPU = gpu_available()

# ---- knobs ----------------------------------------------------------------
FAST = os.environ.get('FAST') == '1'                  # quick smoke test
N_FOLDS = 5
SEEDS = [0] if FAST else [0, 1, 2, 3, 4]              # expensive models (CatBoost)
SEEDS_CHEAP = [0] if FAST else list(range(10))        # cheap models (LightGBM, LogReg, NN)
RUN = dict(cb_cpu=True, cb_cpu_v2feat=True, cb_cpu_slow=True, cb_cpu_deep=True,
           lightgbm=True, lightgbm_xt=True, logreg=True, nn=True)
FULL_REFIT = True        # also train each model on 100% of train for the test predictions
FULL_WEIGHT = 0.5        # test prediction = (1-w) * fold-average + w * full-data models
USE_V3 = True            # add v3's saved predictions to the blend if the files are attached
PREVIOUS_BEST = 0.108765
# ---------------------------------------------------------------------------

print('train:', TRAIN_PATH, '| out:', OUT_DIR, '| GPU:', USE_GPU, '| FAST:', FAST, '| CPUs:', os.cpu_count())

train: /kaggle/input/datasets/jeffmogaka/datadef/Train.csv | out: /kaggle/working | GPU: True | FAST: False | CPUs: 4


## Data and features (same as v2/v3)

In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
y = (train['bank_account'] == 'Yes').astype(int).values
test_ids = test['uniqueid'] + ' x ' + test['country']

CATS = ['country', 'location_type', 'cellphone_access', 'gender_of_respondent',
        'relationship_with_head', 'marital_status', 'education_level', 'job_type']
NUMS = ['household_size', 'age_of_respondent']   # `year` maps 1:1 to country, so it is dropped

X = train[CATS + NUMS].reset_index(drop=True)
X_test = test[CATS + NUMS].reset_index(drop=True)

# Counts use train+test together but never the target, so they cannot leak.
both = pd.concat([X, X_test], ignore_index=True)
def count_of(cols):
    return both.groupby(cols)[cols[0]].transform('size').values

extra = pd.DataFrame({
    'age_cat': both['age_of_respondent'].astype(str),
    'hh_cat': both['household_size'].clip(1, 15).astype(str),
    'cnt_profile': count_of(CATS + NUMS),
    'cnt_ctry_edu_job': count_of(['country', 'education_level', 'job_type']),
    'cnt_ctry_loc_cell': count_of(['country', 'location_type', 'cellphone_access']),
    'cnt_ctry_age': count_of(['country', 'age_of_respondent']),
})
X2 = pd.concat([X, extra.iloc[:len(X)].reset_index(drop=True)], axis=1)
X2_test = pd.concat([X_test, extra.iloc[len(X):].reset_index(drop=True)], axis=1)
CATS2 = CATS + ['age_cat', 'hh_cat']
CNTS = ['cnt_profile', 'cnt_ctry_edu_job', 'cnt_ctry_loc_cell', 'cnt_ctry_age']

print(X2.shape, X2_test.shape, '| positive rate: %.4f' % y.mean())

(23524, 16) (10086, 16) | positive rate: 0.1408


## Cross-validation harness, now with full-data refits
Each model is a function `fit(train_rows, seed)` that returns a predictor `predict(rows)` (`rows=None` means the test set).
Folds are the same as v3 (seeds 0–9), so v3's saved out-of-fold predictions line up row for row.

In [3]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss

FOLDS = {s: list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=s).split(X, y))
         for s in sorted(set(SEEDS) | set(SEEDS_CHEAP))}
ALL_ROWS = np.arange(len(y))
OOF, TEST_FOLD, TEST_FULL, SCORES = {}, {}, {}, {}

def error_rate(p, t=0.5):
    return np.mean((p > t).astype(int) != y)

def run_model(name, fit, seeds=SEEDS):
    t0 = time.time()
    oof = np.zeros((len(seeds), len(y)))
    tst = np.zeros(len(X_test))
    full = np.zeros(len(X_test))
    for k, s in enumerate(seeds):
        for tr_idx, va_idx in FOLDS[s]:
            predict = fit(tr_idx, s)
            oof[k, va_idx] = predict(va_idx)
            tst += predict(None) / (len(seeds) * N_FOLDS)
        if FULL_REFIT:
            full += fit(ALL_ROWS, s)(None) / len(seeds)
    per_seed = [error_rate(o) for o in oof]
    OOF[name], TEST_FOLD[name] = oof.mean(0), tst
    TEST_FULL[name] = full if FULL_REFIT else tst
    SCORES[name] = dict(cv_error=np.mean(per_seed), cv_error_std=np.std(per_seed),
                        logloss=np.mean([log_loss(y, o) for o in oof]),
                        logloss_avg=log_loss(y, OOF[name]), minutes=(time.time() - t0) / 60)
    print(f"{name:14s} logloss {SCORES[name]['logloss']:.5f} (seed-avg {SCORES[name]['logloss_avg']:.5f}) | "
          f"CV error {np.mean(per_seed):.5f} ± {np.std(per_seed):.5f} | "
          f"test fold-vs-full corr {np.corrcoef(tst, TEST_FULL[name])[0, 1]:.4f} | {SCORES[name]['minutes']:.1f} min")

## CatBoost on CPU — four ordered-boosting variants
v3: CPU CatBoost seed-avg logloss **0.27797** (best single model, 5.6 min) vs GPU 0.27843 (15.4 min).
Seed offsets differ from v3, so v3's `catboost_cpu` adds genuinely new seeds when it is reused.

In [4]:
from catboost import CatBoostClassifier

ITERS = lambda n: 300 if FAST else n
CB_VARIANTS = {   # name: (params, features, categorical columns, seed offset)
    'cb_cpu':        (dict(iterations=ITERS(1000), learning_rate=0.03, depth=6, l2_leaf_reg=5,
                           max_ctr_complexity=2), 'X', CATS, 200),
    'cb_cpu_v2feat': (dict(iterations=ITERS(1200), learning_rate=0.03, depth=6, l2_leaf_reg=8,
                           max_ctr_complexity=2, random_strength=1.5), 'X2', CATS2, 300),
    'cb_cpu_slow':   (dict(iterations=ITERS(2000), learning_rate=0.015, depth=6, l2_leaf_reg=5,
                           max_ctr_complexity=2), 'X', CATS, 400),
    'cb_cpu_deep':   (dict(iterations=ITERS(1000), learning_rate=0.03, depth=7, l2_leaf_reg=10,
                           max_ctr_complexity=3, random_strength=1.5), 'X', CATS, 500),
}
FEATS = {'X': (X, X_test), 'X2': (X2, X2_test)}

def catboost_fit(params, feats, cats, offset):
    Xa, Xa_test = FEATS[feats]
    def fit(tr_idx, seed):
        m = CatBoostClassifier(**params, boosting_type='Ordered', border_count=254, thread_count=-1,
                               random_seed=seed + offset, verbose=0, allow_writing_files=False)
        m.fit(Xa.iloc[tr_idx], y[tr_idx], cat_features=cats)
        return lambda rows: m.predict_proba(Xa_test if rows is None else Xa.iloc[rows])[:, 1]
    return fit

for name, (params, feats, cats, offset) in CB_VARIANTS.items():
    if RUN[name]:
        run_model(name, catboost_fit(params, feats, cats, offset))

cb_cpu         logloss 0.27925 (seed-avg 0.27798) | CV error 0.11178 ± 0.00025 | test fold-vs-full corr 0.9993 | 10.9 min
cb_cpu_v2feat  logloss 0.27980 (seed-avg 0.27828) | CV error 0.11225 ± 0.00047 | test fold-vs-full corr 0.9990 | 17.2 min
cb_cpu_slow    logloss 0.27905 (seed-avg 0.27781) | CV error 0.11183 ± 0.00030 | test fold-vs-full corr 0.9993 | 21.9 min
cb_cpu_deep    logloss 0.27890 (seed-avg 0.27764) | CV error 0.11174 ± 0.00039 | test fold-vs-full corr 0.9991 | 15.2 min


## LightGBM — standard and extra-trees variant (unchanged from v3)

In [5]:
import lightgbm as lgb

def as_category(df):
    df = df.copy()
    for c in CATS:
        df[c] = pd.Categorical(df[c], categories=sorted(set(X2[c]) | set(X2_test[c])))
    return df

LGB_COLS = CATS + NUMS + CNTS
Xc, Xc_test = as_category(X2[LGB_COLS]), as_category(X2_test[LGB_COLS])

LGB_PARAMS = dict(n_estimators=150 if FAST else 700, learning_rate=0.02, num_leaves=24, min_child_samples=40,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.7, cat_smooth=20, cat_l2=10,
                  reg_lambda=3, verbose=-1, n_jobs=-1)
LGB_XT_PARAMS = dict(LGB_PARAMS, n_estimators=150 if FAST else 1200, extra_trees=True,
                     num_leaves=31, min_child_samples=25)

def lgb_fit(params):
    def fit(tr_idx, seed):
        m = lgb.LGBMClassifier(**params, random_state=seed)
        m.fit(Xc.iloc[tr_idx], y[tr_idx])
        return lambda rows: m.predict_proba(Xc_test if rows is None else Xc.iloc[rows])[:, 1]
    return fit

if RUN['lightgbm']:
    run_model('lightgbm', lgb_fit(LGB_PARAMS), SEEDS_CHEAP)
if RUN['lightgbm_xt']:
    run_model('lightgbm_xt', lgb_fit(LGB_XT_PARAMS), SEEDS_CHEAP)

lightgbm       logloss 0.28158 (seed-avg 0.27988) | CV error 0.11274 ± 0.00063 | test fold-vs-full corr 0.9998 | 1.6 min
lightgbm_xt    logloss 0.28171 (seed-avg 0.28013) | CV error 0.11372 ± 0.00054 | test fold-vs-full corr 0.9998 | 3.3 min


## Logistic regression on pairwise + three-way crosses
v3's grid confirmed `C=0.02` is the minimum (0.014: 0.28019, 0.02: 0.28011, 0.03: 0.28052), so it is fixed.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

TRIPLES = [('country', 'education_level', 'job_type'),
           ('country', 'location_type', 'cellphone_access'),
           ('country', 'age_b', 'gender_of_respondent'),
           ('country', 'education_level', 'age_b'),
           ('country', 'relationship_with_head', 'marital_status')]

def crosses(df):
    d = df[CATS].copy()
    d['age_b'] = pd.cut(df['age_of_respondent'], [0, 20, 25, 30, 35, 40, 50, 60, 70, 200]).astype(str)
    d['hh_b'] = df['household_size'].clip(1, 12).astype(str)
    base = list(d.columns)
    for i in range(len(base)):
        for j in range(i + 1, len(base)):
            d[base[i] + '*' + base[j]] = d[base[i]] + '|' + d[base[j]]
    for a, b, c in TRIPLES:
        d[f'{a}*{b}*{c}'] = d[a] + '|' + d[b] + '|' + d[c]
    return d

Xx, Xx_test = crosses(X), crosses(X_test)
LR_C = 0.02

def lr_fit(tr_idx, seed):
    enc = OneHotEncoder(handle_unknown='ignore', min_frequency=5)
    m = LogisticRegression(C=LR_C, max_iter=3000).fit(enc.fit_transform(Xx.iloc[tr_idx]), y[tr_idx])
    return lambda rows: m.predict_proba(enc.transform(Xx_test if rows is None else Xx.iloc[rows]))[:, 1]

if RUN['logreg']:
    run_model('logreg_cross', lr_fit, SEEDS_CHEAP)

logreg_cross   logloss 0.28022 (seed-avg 0.27957) | CV error 0.11204 ± 0.00054 | test fold-vs-full corr 0.9998 | 1.7 min


## Neural net with entity embeddings (GPU, same as v3)

In [7]:
import torch
import torch.nn as nn

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EMB_COLS = CATS2
NN_EPOCHS, NN_BS = (3 if FAST else 10), 256

categories = {c: pd.Categorical(pd.concat([X2[c], X2_test[c]])).categories for c in EMB_COLS}

def nn_encode(df):
    cat = np.column_stack([pd.Categorical(df[c], categories=categories[c]).codes for c in EMB_COLS]).astype(np.int64)
    num = np.column_stack([df['age_of_respondent'], df['household_size']] +
                          [np.log1p(df[c]) for c in CNTS]).astype(np.float32)
    return cat, num

N_cat, N_num = nn_encode(X2)
T_cat, T_num = nn_encode(X2_test)
mu, sd = np.vstack([N_num, T_num]).mean(0), np.vstack([N_num, T_num]).std(0) + 1e-6
N_num, T_num = (N_num - mu) / sd, (T_num - mu) / sd

class EmbNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.embs = nn.ModuleList([nn.Embedding(len(categories[c]), min(8, (len(categories[c]) + 1) // 2 + 1))
                                   for c in EMB_COLS])
        d = sum(e.embedding_dim for e in self.embs) + N_num.shape[1]
        self.mlp = nn.Sequential(
            nn.Dropout(0.15),
            nn.Linear(d, 128), nn.BatchNorm1d(128), nn.SiLU(), nn.Dropout(0.35),
            nn.Linear(128, 32), nn.BatchNorm1d(32), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(32, 1))

    def forward(self, c, x):
        h = torch.cat([e(c[:, i]) for i, e in enumerate(self.embs)] + [x], dim=1)
        return self.mlp(h).squeeze(1)

def to_t(a):
    return torch.from_numpy(a).to(DEVICE)

@torch.no_grad()
def nn_predict(model, cat, num):
    model.eval()
    return torch.sigmoid(model(to_t(cat), to_t(num))).cpu().numpy()

def nn_fit(tr_idx, seed):
    torch.manual_seed(seed)
    model = EmbNet().to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)
    steps = len(tr_idx) // NN_BS
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, total_steps=NN_EPOCHS * steps)
    c_tr, x_tr = to_t(N_cat[tr_idx]), to_t(N_num[tr_idx])
    y_tr = torch.tensor(y[tr_idx], dtype=torch.float32, device=DEVICE)
    loss_fn = nn.BCEWithLogitsLoss()
    g = torch.Generator(device='cpu').manual_seed(seed)
    for _ in range(NN_EPOCHS):
        model.train()
        perm = torch.randperm(len(tr_idx), generator=g).to(DEVICE)
        for b in range(steps):                       # drops the last partial batch (BatchNorm-safe)
            i = perm[b * NN_BS:(b + 1) * NN_BS]
            opt.zero_grad()
            loss_fn(model(c_tr[i], x_tr[i]), y_tr[i]).backward()
            opt.step()
            sched.step()
    return lambda rows: (nn_predict(model, T_cat, T_num) if rows is None
                         else nn_predict(model, N_cat[rows], N_num[rows]))

if RUN['nn']:
    print('NN device:', DEVICE)
    run_model('embed_nn', nn_fit, SEEDS_CHEAP)

NN device: cuda
embed_nn       logloss 0.28405 (seed-avg 0.28128) | CV error 0.11361 ± 0.00050 | test fold-vs-full corr 0.9996 | 2.6 min


## Save every model's predictions

In [8]:
pd.DataFrame({'y': y, **OOF}).to_csv(f'{OUT_DIR}/oof_v4.csv', index=False)
pd.DataFrame({'unique_id': test_ids, **TEST_FOLD}).to_csv(f'{OUT_DIR}/test_probabilities_v4_fold.csv', index=False)
pd.DataFrame({'unique_id': test_ids, **TEST_FULL}).to_csv(f'{OUT_DIR}/test_probabilities_v4_full.csv', index=False)
print('saved', list(OOF))

saved ['cb_cpu', 'cb_cpu_v2feat', 'cb_cpu_slow', 'cb_cpu_deep', 'lightgbm', 'lightgbm_xt', 'logreg_cross', 'embed_nn']


## Add v3's models (if attached)
v3's files have fold-only test predictions, so they are used as-is in both test variants.
Columns whose name already exists in v4 (v4 retrained them with identical seeds) are skipped.

In [9]:
oof3_path, tst3_path = find_file('oof_v3.csv'), find_file('test_probabilities_v3.csv')
if USE_V3 and oof3_path and tst3_path:
    oof3, tst3 = pd.read_csv(oof3_path), pd.read_csv(tst3_path)
    assert (oof3['y'].values == y).all(), 'oof_v3.csv rows do not match Train.csv'
    assert (tst3['unique_id'].values == test_ids.values).all(), 'test_probabilities_v3.csv rows do not match Test.csv'
    for c in oof3.columns.drop('y'):
        if c in OOF:
            continue
        name = c + '@v3'
        OOF[name], TEST_FOLD[name], TEST_FULL[name] = oof3[c].values, tst3[c].values, tst3[c].values
        SCORES[name] = dict(cv_error=error_rate(OOF[name]), cv_error_std=np.nan, logloss=np.nan,
                            logloss_avg=log_loss(y, OOF[name]), minutes=0.0)
        print(f'added {name:16s} seed-avg logloss {SCORES[name]["logloss_avg"]:.5f}')
else:
    print('v3 predictions not attached - blending v4 models only.')

v3 predictions not attached - blending v4 models only.


## Combiners, judged on held-out logloss
`prob_blend` (v2's and v3's winner) now fits its weights with SLSQP on the simplex; `logit_blend` is the runner-up from v3.
Each is fitted on 4/5 of the OOF rows and scored on the other 1/5, over 3 different splits.

In [10]:
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression

names = list(OOF)
P = np.column_stack([OOF[n] for n in names])
T_FOLD = np.column_stack([TEST_FOLD[n] for n in names])
T_FULL = np.column_stack([TEST_FULL[n] for n in names])
T_MIX = (1 - FULL_WEIGHT) * T_FOLD + FULL_WEIGHT * T_FULL

def logit(p):
    p = np.clip(p, 1e-5, 1 - 1e-5)
    return np.log(p / (1 - p))

class ProbBlend:                       # weighted average of probabilities, weights >= 0 summing to 1
    def fit(self, F, t):
        F = np.clip(F, 1e-6, 1 - 1e-6)
        def loss(w):
            p = np.clip(F @ w, 1e-6, 1 - 1e-6)
            return -np.mean(t * np.log(p) + (1 - t) * np.log(1 - p))
        def grad(w):
            p = np.clip(F @ w, 1e-6, 1 - 1e-6)
            return -F.T @ (t / p - (1 - t) / (1 - p)) / len(t)
        k = F.shape[1]
        r = minimize(loss, np.ones(k) / k, jac=grad, method='SLSQP', bounds=[(0, 1)] * k,
                     constraints=[dict(type='eq', fun=lambda w: w.sum() - 1, jac=lambda w: np.ones(k))],
                     options=dict(maxiter=500, ftol=1e-10))
        self.w = np.clip(r.x, 0, None) / np.clip(r.x, 0, None).sum()
        return self
    def predict(self, F):
        return F @ self.w

class LogitBlend:
    def fit(self, F, t):
        self.m = LogisticRegression(C=1.0, max_iter=5000).fit(logit(F), t); return self
    def predict(self, F):
        return self.m.predict_proba(logit(F))[:, 1]

CANDIDATES = {'prob_blend': ProbBlend, 'logit_blend': LogitBlend}

HELD_OOF = {k: np.zeros(len(y)) for k in CANDIDATES}
META_SEEDS = [2024, 7, 99]
for ms in META_SEEDS:
    for tr, va in StratifiedKFold(5, shuffle=True, random_state=ms).split(P, y):
        for k, make in CANDIDATES.items():
            HELD_OOF[k][va] += make().fit(P[tr], y[tr]).predict(P[va]) / len(META_SEEDS)
for k in CANDIDATES:
    print(f'{k:12s} held-out logloss {log_loss(y, HELD_OOF[k]):.5f} | error {error_rate(HELD_OOF[k]):.5f}')

FITTED = {k: make().fit(P, y) for k, make in CANDIDATES.items()}
ranked = sorted(CANDIDATES, key=lambda k: log_loss(y, HELD_OOF[k]))
print('\nranking by held-out logloss:', ranked)
print('prob_blend weights:', {n: round(float(w), 3) for n, w in zip(names, FITTED['prob_blend'].w) if w > 0.0005})

prob_blend   held-out logloss 0.27729 | error 0.11193
logit_blend  held-out logloss 0.27744 | error 0.11163

ranking by held-out logloss: ['prob_blend', 'logit_blend']
prob_blend weights: {'cb_cpu_slow': 0.098, 'cb_cpu_deep': 0.473, 'lightgbm': 0.151, 'lightgbm_xt': 0.084, 'logreg_cross': 0.12, 'embed_nn': 0.074}


## Cutoff check
Only moves away from 0.5 if the gain beats a noise floor of 0.001 (~24 training rows). It has stayed at 0.5 in every run so far.

In [11]:
grid = np.round(np.arange(0.40, 0.601, 0.01), 2)
THRESH = {}
for k in CANDIDATES:
    o = HELD_OOF[k]
    errs = np.array([error_rate(o, t) for t in grid])
    THRESH[k] = grid[errs.argmin()] if errs.min() < error_rate(o) - 0.001 else 0.5
    print(f'{k:12s} best cutoff {grid[errs.argmin()]} (err {errs.min():.5f}) -> using {THRESH[k]}')

prob_blend   best cutoff 0.48 (err 0.11142) -> using 0.5
logit_blend  best cutoff 0.48 (err 0.11116) -> using 0.5


## Leaderboard of models and combiners (sorted by logloss)

In [12]:
board = pd.DataFrame(SCORES).T[['logloss', 'logloss_avg', 'cv_error', 'cv_error_std', 'minutes']]
for k in CANDIDATES:
    board.loc['>> ' + k] = [np.nan, log_loss(y, HELD_OOF[k]), error_rate(HELD_OOF[k], THRESH[k]), np.nan, np.nan]
display(board.sort_values('logloss_avg').round(5))
print('Reference held-out logloss: v2 blend 0.27754 (LB 1097 wrong rows), v3 prob_blend 0.27746 (LB 1101).')
print('Differences under ~0.0002 are within leaderboard noise.')

,logloss,logloss_avg,cv_error,cv_error_std,minutes
>> prob_blend,NaN,0.27729,0.11193,NaN,NaN
>> logit_blend,NaN,0.27744,0.11163,NaN,NaN
cb_cpu_deep,0.27890,0.27764,0.11174,0.00039,15.24886
cb_cpu_slow,0.27905,0.27781,0.11183,0.00030,21.92100
cb_cpu,0.27925,0.27798,0.11178,0.00025,10.86820
cb_cpu_v2feat,0.27980,0.27828,0.11225,0.00047,17.23689
logreg_cross,0.28022,0.27957,0.11204,0.00054,1.69987
lightgbm,0.28158,0.27988,0.11274,0.00063,1.60461
lightgbm_xt,0.28171,0.28013,0.11372,0.00054,3.28215
embed_nn,0.28405,0.28128,0.11361,0.00050,2.61514


Reference held-out logloss: v2 blend 0.27754 (LB 1097 wrong rows), v3 prob_blend 0.27746 (LB 1101).
Differences under ~0.0002 are within leaderboard noise.


## Write submissions
- `submission_v4_best.csv`: best combiner, test predictions mixing fold models and full-data models. (primary submission)
- `submission_v4_foldonly.csv`: the same combiner with fold-only test predictions (the v1–v3 way). Comparing the two isolates the effect of the full-data refits.
- `submission_v4_second.csv`: the runner-up combiner (fold+full), as a spare.

In [13]:
def save(name, probs, t=0.5):
    sub = pd.DataFrame({'unique_id': test_ids, 'bank_account': (probs > t).astype(int)})
    path = f'{OUT_DIR}/submission_v4_{name}.csv'
    sub.to_csv(path, index=False)
    print(f'{path}: {sub.bank_account.sum()} ones / {len(sub)} rows')
    return sub

best, second = ranked[0], ranked[1]
print('best =', best, '| second =', second)
SUBS = {
    'best': save('best', FITTED[best].predict(T_MIX), THRESH[best]),
    'foldonly': save('foldonly', FITTED[best].predict(T_FOLD), THRESH[best]),
    'second': save('second', FITTED[second].predict(T_MIX), THRESH[second]),
}

def labels(df):
    return df.set_index('unique_id')['bank_account'].reindex(test_ids).values

print(f"best vs foldonly: {(labels(SUBS['best']) != labels(SUBS['foldonly'])).sum()} rows differ")
for prev_name in ['submission_v2_recommended.csv', 'submission_v2_blend.csv', 'submission_v3_best.csv']:
    prev_path = find_file(prev_name)
    if prev_path:
        prev = labels(pd.read_csv(prev_path))
        for k, sub in SUBS.items():
            print(f'{k:9s} vs {prev_name}: {(labels(sub) != prev).sum()} rows differ')

best = prob_blend | second = logit_blend
/kaggle/working/submission_v4_best.csv: 750 ones / 10086 rows
/kaggle/working/submission_v4_foldonly.csv: 753 ones / 10086 rows
/kaggle/working/submission_v4_second.csv: 772 ones / 10086 rows
best vs foldonly: 13 rows differ
